<a href="https://colab.research.google.com/github/rlaqhtjd-2026/group-8-surveying2/blob/main/02_%EA%B0%9C%EC%9D%B8%EB%B6%84%EC%84%9D/%EC%84%B8%EB%B6%80%EB%AA%A9%EC%A0%81(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

설치 및 초기화

In [18]:
# 필요한 패키지 설치
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "earthengine-api", "geemap", "pandas",
    "openpyxl", "ipywidgets", "koreanize-matplotlib"
])

import ee
import geemap
import pandas as pd
import numpy as np
import ipywidgets as widgets
import ipyleaflet
import traceback
import io
import requests
import math

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image

from pathlib import Path
from google.colab import files, output
from IPython.display import display, clear_output

import koreanize_matplotlib

# Colab에서 지도 위젯이 정상적으로 보이도록 설정
output.enable_custom_widget_manager()

# Google Earth Engine 프로젝트
PROJECT = "class-2026-508800"

# Sentinel-2 Level-2A 자료
COLLECTION_ID = "COPERNICUS/S2_SR_HARMONIZED"

# 분석 기간
START_DATE = "2026-05-01"
END_DATE = "2026-10-01"

# 전체 영상에 적용할 구름 비율 조건
MAX_CLOUD_PERCENT = 20.0

# NDCI-클로로필-a 회귀식
SLOPE = 191.688874
INTERCEPT = 23.757085

# 현재 채수지점 좌표
SITE_LON = 128.9523888889
SITE_LAT = 35.33525

# 수체 판별 기준
MNDWI_THRESHOLD = 0.18
NDWI_THRESHOLD = 0.08
MAX_NIR = 0.15

# 결과 저장 폴더
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Google Earth Engine 초기화
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

# 채수지점
site_point = ee.Geometry.Point([SITE_LON, SITE_LAT])

# 채수지점 중심의 약 20km × 20km 분석 범위
lat_offset = 10.0 / 111.0
lon_offset = 10.0 / 90.7

roi = ee.Geometry.Rectangle([
    SITE_LON - lon_offset,
    SITE_LAT - lat_offset,
    SITE_LON + lon_offset,
    SITE_LAT + lat_offset
])

# 이전 계산 결과 초기화
analysis_cache = {}
saved_map_state = {}

print("초기화 완료")
print("기준: 현재 채수지점 좌표가 속한 20m 픽셀 1개")
print("버퍼 평균은 사용하지 않습니다.")
print(f"회귀식: Chl-a = {SLOPE} × NDCI + {INTERCEPT}")
print("농도·편차 단위: mg/m³")

초기화 완료
기준: 현재 채수지점 좌표가 속한 20m 픽셀 1개
버퍼 평균은 사용하지 않습니다.
회귀식: Chl-a = 191.688874 × NDCI + 23.757085
농도·편차 단위: mg/m³


전체 영상 조회

In [19]:
# Sentinel-2 영상 조회
s2 = (
    ee.ImageCollection(COLLECTION_ID)
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            MAX_CLOUD_PERCENT
        )
    )
    .sort("system:time_start")
)

# 조회된 영상 수 확인
image_count = int(s2.size().getInfo())

if image_count == 0:
    raise ValueError("조건에 해당하는 영상이 없습니다.")

# 영상 정보 가져오기
metadata = ee.List(
    s2.toList(image_count).map(
        lambda obj: ee.Dictionary({
            "Image_ID": ee.Image(obj).id(),
            "Image_Index": ee.Image(obj).get("system:index"),
            "Date_Time_KST": ee.Image(obj).date().format(
                "YYYY-MM-dd HH:mm",
                "Asia/Seoul"
            ),
            "Cloud_Percent_Global": ee.Image(obj).get(
                "CLOUDY_PIXEL_PERCENTAGE"
            ),
            "MGRS_Tile": ee.Image(obj).get("MGRS_TILE"),
            "Satellite": ee.Image(obj).get("SPACECRAFT_NAME")
        })
    )
).getInfo()

# 전체 영상 목록
candidate_image_table = (
    pd.DataFrame(metadata)
    .sort_values(["Date_Time_KST", "Image_Index"])
    .reset_index(drop=True)
)

candidate_image_table.insert(
    0, "No", range(1, len(candidate_image_table) + 1)
)

candidate_image_table["Date_KST"] = (
    candidate_image_table["Date_Time_KST"].str[:10]
)

candidate_image_table["Analysis_Status"] = "미처리"

# 영상 ID가 짧은 형태로 저장된 경우를 대비해
# 전체 경로로 통일
candidate_image_table["Image_ID"] = (
    candidate_image_table["Image_ID"]
    .astype(str)
    .apply(
        lambda value: value
        if value.startswith(COLLECTION_ID)
        else f"{COLLECTION_ID}/{value}"
    )
)

print(f"조회 영상: {len(candidate_image_table)}장")
print(
    f"서로 다른 촬영 날짜: "
    f"{candidate_image_table['Date_KST'].nunique()}일"
)
print("동일 날짜의 서로 다른 타일은 별도 영상입니다.")

print("\n첫 영상의 전체 ID 확인")
print(candidate_image_table.loc[0, "Image_ID"])

display(candidate_image_table[[
    "No",
    "Date_Time_KST",
    "Satellite",
    "MGRS_Tile",
    "Cloud_Percent_Global",
    "Image_Index"
]])

조회 영상: 19장
서로 다른 촬영 날짜: 13일
동일 날짜의 서로 다른 타일은 별도 영상입니다.

첫 영상의 전체 ID 확인
COPERNICUS/S2_SR_HARMONIZED/20260508T020649_20260508T021227_T52SDE


,No,Date_Time_KST,Satellite,MGRS_Tile,Cloud_Percent_Global,Image_Index
0,1,2026-05-08 11:17,Sentinel-2B,52SDE,5.836149,20260508T020649_20260508T021227_T52SDE
1,2,2026-05-08 11:17,Sentinel-2B,52SEE,5.079857,20260508T020649_20260508T021227_T52SEE
2,3,2026-05-13 11:17,Sentinel-2C,52SDE,1.342889,20260513T020651_20260513T021205_T52SDE
3,4,2026-05-15 11:17,Sentinel-2A,52SDE,0.266087,20260515T021121_20260515T021122_T52SDE
4,5,2026-05-18 11:17,Sentinel-2B,52SDE,6.823023,20260518T020649_20260518T021212_T52SDE
5,6,2026-05-18 11:17,Sentinel-2B,52SEE,9.440792,20260518T020649_20260518T021212_T52SEE
6,7,2026-06-12 11:17,Sentinel-2C,52SDE,0.148321,20260612T020651_20260612T021313_T52SDE
7,8,2026-06-12 11:17,Sentinel-2C,52SEE,0.035192,20260612T020651_20260612T021313_T52SEE
8,9,2026-07-22 11:17,Sentinel-2C,52SEE,8.951852,20260722T020651_20260722T021351_T52SEE
9,10,2026-08-01 11:17,Sentinel-2C,52SDE,17.945518,20260801T020651_20260801T021347_T52SDE


단일 픽셀 분석 함수

In [23]:
def process_scene(image_id):
    # image_id가 영상 이름만 들어와도
    # 전체 자료 경로를 자동으로 만들도록 수정
    image_id = str(image_id)

    if not image_id.startswith(COLLECTION_ID):
        image_id = f"{COLLECTION_ID}/{image_id}"

    image = ee.Image(image_id)
    scl = image.select("SCL")

    # 선택 영상의 B5 원래 20m 격자
    grid = image.select("B5").projection()

    # 예비 품질 마스크
    # 4: 식생, 5: 비식생, 6: 수면, 7: 미분류
    clear_mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
    )

    # 정수 저장값을 반사도로 변환
    reflectance = (
        image.select(["B3", "B4", "B5", "B8", "B11"])
        .multiply(0.0001)
        .updateMask(clear_mask)
    )

    # 10m 밴드를 B5의 20m 격자에 맞추기
    def aggregate_20m(band):
        return (
            reflectance.select(band)
            .reduceResolution(
                reducer=ee.Reducer.mean(),
                maxPixels=64
            )
            .reproject(crs=grid)
        )

    green = aggregate_20m("B3")
    red = aggregate_20m("B4")
    nir = aggregate_20m("B8")

    red_edge = (
        reflectance.select("B5")
        .reproject(crs=grid)
    )

    swir1 = (
        reflectance.select("B11")
        .reproject(crs=grid)
    )

    # 수체 판별 지수
    mndwi_den = green.add(swir1)
    ndwi_den = green.add(nir)

    mndwi = (
        green.subtract(swir1)
        .divide(mndwi_den)
        .updateMask(mndwi_den.gt(0))
    )

    ndwi = (
        green.subtract(nir)
        .divide(ndwi_den)
        .updateMask(ndwi_den.gt(0))
    )

    # 수면 분류 또는 수체 지수 조건을 만족하는 픽셀 유지
    water_mask = (
        scl.eq(6)
        .Or(
            mndwi.gt(MNDWI_THRESHOLD)
            .And(ndwi.gt(NDWI_THRESHOLD))
            .And(nir.lt(MAX_NIR))
        )
        .And(clear_mask)
    )

    # NDCI 계산
    denominator = red_edge.add(red)

    ndci = (
        red_edge.subtract(red)
        .divide(denominator)
        .updateMask(
            denominator.gt(0)
            .And(red.gte(0))
            .And(red_edge.gte(0))
        )
        .updateMask(water_mask)
        .rename("NDCI")
        .clip(roi)
    )

    # 회귀식을 이용한 추정 클로로필-a
    chla = (
        ndci.multiply(SLOPE)
        .add(INTERCEPT)
        .rename("Chla_est_mg_m3")
    )

    # 버퍼 없이 채수지점 좌표가 속한 픽셀 1개 추출
    # B5의 원래 20m 해상도를 사용
    site_stats = ndci.addBands(chla).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=site_point,
        scale=20,
        maxPixels=1000
    ).getInfo()

    site_ndci = site_stats.get("NDCI")
    site_chla = site_stats.get("Chla_est_mg_m3")

    # 기준 픽셀의 값이 있는지 확인
    site_valid = (
        site_ndci is not None
        and site_chla is not None
    )

    if site_valid:
        site_ndci = float(site_ndci)
        site_chla = float(site_chla)

        # 편차 = 주변 픽셀 추정 농도 - 채수지점 픽셀 추정 농도
        deviation = (
            chla.subtract(site_chla)
            .rename("Deviation_mg_m3")
        )

    else:
        # 기준 픽셀이 결측이면 편차도 결측 처리
        # 주변 픽셀 값으로 대체하지 않음
        deviation = (
            chla.multiply(0)
            .updateMask(ee.Image.constant(0))
            .rename("Deviation_mg_m3")
        )

    # 채수지점 좌표가 속한 기준 픽셀 경계
    # 지도 표시용으로 약 20m 크기 영역을 사용
    # 값 추출은 버퍼 평균이 아니라 단일 픽셀 방식
    site_pixel_buffer = site_point.buffer(10)

    pixel_geometry = site_pixel_buffer.bounds()

    stack = (
        ndci
        .addBands(chla)
        .addBands(deviation)
    )

    return {
        "raw": image,
        "grid": grid,
        "ndci": ndci,
        "chla": chla,
        "deviation": deviation,
        "stack": stack,
        "site_ndci": site_ndci,
        "site_chla": site_chla,
        "site_valid": site_valid,
        "site_pixel_geometry": pixel_geometry
    }


def get_result(index):
    image_id = candidate_image_table.loc[
        index, "Image_ID"
    ]

    # 이미 계산한 영상은 저장된 결과 사용
    if image_id not in analysis_cache:
        result = process_scene(image_id)
        analysis_cache[image_id] = result

    return analysis_cache[image_id]


print("단일 픽셀 분석 함수 준비 완료")
print("기준: 현재 채수지점 좌표가 속한 20m 픽셀 1개")
print("채수지점 주변의 버퍼 평균은 사용하지 않습니다.")
print("기준 픽셀이 결측이면 주변 픽셀로 대체하지 않습니다.")
print("채수지점 픽셀 값 추출 방식: scale=20")

단일 픽셀 분석 함수 준비 완료
기준: 현재 채수지점 좌표가 속한 20m 픽셀 1개
채수지점 주변의 버퍼 평균은 사용하지 않습니다.
기준 픽셀이 결측이면 주변 픽셀로 대체하지 않습니다.
채수지점 픽셀 값 추출 방식: scale=20


**기준 픽셀 값이 있는 영상만 선정**

이 단계에서 각 영상의 기준 픽셀을 실제로 검사합니다. 값이 있는 영상만 이후 지도와 엑셀에 사용합니다.

In [24]:
# 이전 계산 결과 초기화
analysis_cache = {}

# 영상 선정 결과 저장
selection_records = []
accepted_indices = []

print("현재 채수지점의 단일 픽셀 값 확인을 시작합니다.")
print("구름·수면 마스크를 적용한 뒤 값이 있는 영상만 선정합니다.")
print("영상 수에 따라 시간이 걸릴 수 있습니다.\n")

total = len(candidate_image_table)

for position, (index, row) in enumerate(
    candidate_image_table.iterrows(), start=1
):
    record = {
        "Image_ID": row["Image_ID"],
        "Image_Index": row["Image_Index"],
        "Date_Time_KST": row["Date_Time_KST"],
        "선정_여부": False,
        "검사_결과": "",
        "기준_픽셀_NDCI": None,
        "기준_픽셀_추정농도_mg_m3": None,
        "오류_내용": ""
    }

    print(
        f"[{position}/{total}] "
        f"{row['Date_Time_KST']} | {row['Image_Index']}",
        end=" → "
    )

    try:
        image_id = row["Image_ID"]

        if image_id not in analysis_cache:
            analysis_cache[image_id] = process_scene(image_id)

        result = analysis_cache[image_id]

        valid = (
            result["site_valid"]
            and result["site_ndci"] is not None
            and result["site_chla"] is not None
            and math.isfinite(float(result["site_ndci"]))
            and math.isfinite(float(result["site_chla"]))
        )

        if valid:
            accepted_indices.append(index)

            record["선정_여부"] = True
            record["검사_결과"] = "선정"
            record["기준_픽셀_NDCI"] = float(
                result["site_ndci"]
            )
            record["기준_픽셀_추정농도_mg_m3"] = float(
                result["site_chla"]
            )

            print(
                "선정 "
                f"(추정 농도 {result['site_chla']:.2f} mg/m³)"
            )

        else:
            record["검사_결과"] = "제외: 기준 픽셀 결측"
            print("제외: 기준 픽셀 결측")

    except Exception as error:
        record["검사_결과"] = "판정 보류: 처리 오류"
        record["오류_내용"] = str(error)

        print("판정 보류: 처리 오류")

    selection_records.append(record)

# 영상 선정 내역
selection_table = pd.DataFrame(selection_records)

# 이후 지도·엑셀에서 사용할 영상은 기준 픽셀 값이 있는 영상만
image_table = (
    candidate_image_table.loc[accepted_indices]
    .copy()
    .reset_index(drop=True)
)

if image_table.empty:
    display(selection_table)
    raise ValueError(
        "선정된 영상이 없습니다. "
        "기준 픽셀 결측과 처리 오류 내역을 확인하세요."
    )

image_table["No"] = range(1, len(image_table) + 1)
image_table["Analysis_Status"] = "기준 픽셀 유효"

image_table["Site_Pixel_Chla_mg_m3"] = [
    analysis_cache[image_id]["site_chla"]
    for image_id in image_table["Image_ID"]
]

print("\n영상 선정 완료")
print(f"검사 대상: {len(candidate_image_table)}장")
print(f"선정 영상: {len(image_table)}장")
print(
    "처리 오류: "
    f"{(selection_table['검사_결과'] == '판정 보류: 처리 오류').sum()}장"
)

display(image_table[[
    "No",
    "Date_Time_KST",
    "Image_Index",
    "Site_Pixel_Chla_mg_m3"
]])

display(selection_table)

현재 채수지점의 단일 픽셀 값 확인을 시작합니다.
구름·수면 마스크를 적용한 뒤 값이 있는 영상만 선정합니다.
영상 수에 따라 시간이 걸릴 수 있습니다.

[1/19] 2026-05-08 11:17 | 20260508T020649_20260508T021227_T52SDE → 선정 (추정 농도 17.66 mg/m³)
[2/19] 2026-05-08 11:17 | 20260508T020649_20260508T021227_T52SEE → 제외: 기준 픽셀 결측
[3/19] 2026-05-13 11:17 | 20260513T020651_20260513T021205_T52SDE → 선정 (추정 농도 -167.93 mg/m³)
[4/19] 2026-05-15 11:17 | 20260515T021121_20260515T021122_T52SDE → 선정 (추정 농도 20.44 mg/m³)
[5/19] 2026-05-18 11:17 | 20260518T020649_20260518T021212_T52SDE → 선정 (추정 농도 5.27 mg/m³)
[6/19] 2026-05-18 11:17 | 20260518T020649_20260518T021212_T52SEE → 제외: 기준 픽셀 결측
[7/19] 2026-06-12 11:17 | 20260612T020651_20260612T021313_T52SDE → 선정 (추정 농도 45.37 mg/m³)
[8/19] 2026-06-12 11:17 | 20260612T020651_20260612T021313_T52SEE → 제외: 기준 픽셀 결측
[9/19] 2026-07-22 11:17 | 20260722T020651_20260722T021351_T52SEE → 제외: 기준 픽셀 결측
[10/19] 2026-08-01 11:17 | 20260801T020651_20260801T021347_T52SDE → 선정 (추정 농도 34.97 mg/m³)
[11/19] 2026-08-03 11:17 | 20260803T021131_20260803

,No,Date_Time_KST,Image_Index,Site_Pixel_Chla_mg_m3
0,1,2026-05-08 11:17,20260508T020649_20260508T021227_T52SDE,17.661615
1,2,2026-05-13 11:17,20260513T020651_20260513T021205_T52SDE,-167.931789
2,3,2026-05-15 11:17,20260515T021121_20260515T021122_T52SDE,20.438481
3,4,2026-05-18 11:17,20260518T020649_20260518T021212_T52SDE,5.274687
4,5,2026-06-12 11:17,20260612T020651_20260612T021313_T52SDE,45.370056
5,6,2026-08-01 11:17,20260801T020651_20260801T021347_T52SDE,34.974507
6,7,2026-08-06 11:17,20260806T020649_20260806T021309_T52SDE,42.788067
7,8,2026-09-12 11:17,20260912T021121_20260912T021121_T52SDE,54.618559
8,9,2026-09-30 11:17,20260930T020651_20260930T021242_T52SDE,51.559288


,Image_ID,Image_Index,Date_Time_KST,선정_여부,검사_결과,기준_픽셀_NDCI,기준_픽셀_추정농도_mg_m3,오류_내용
0,COPERNICUS/S2_SR_HARMONIZED/20260508T020649_20...,20260508T020649_20260508T021227_T52SDE,2026-05-08 11:17,True,선정,-0.031799,17.661615,
1,COPERNICUS/S2_SR_HARMONIZED/20260508T020649_20...,20260508T020649_20260508T021227_T52SEE,2026-05-08 11:17,False,제외: 기준 픽셀 결측,NaN,NaN,
2,COPERNICUS/S2_SR_HARMONIZED/20260513T020651_20...,20260513T020651_20260513T021205_T52SDE,2026-05-13 11:17,True,선정,-1.000000,-167.931789,
3,COPERNICUS/S2_SR_HARMONIZED/20260515T021121_20...,20260515T021121_20260515T021122_T52SDE,2026-05-15 11:17,True,선정,-0.017312,20.438481,
4,COPERNICUS/S2_SR_HARMONIZED/20260518T020649_20...,20260518T020649_20260518T021212_T52SDE,2026-05-18 11:17,True,선정,-0.096419,5.274687,
5,COPERNICUS/S2_SR_HARMONIZED/20260518T020649_20...,20260518T020649_20260518T021212_T52SEE,2026-05-18 11:17,False,제외: 기준 픽셀 결측,NaN,NaN,
6,COPERNICUS/S2_SR_HARMONIZED/20260612T020651_20...,20260612T020651_20260612T021313_T52SDE,2026-06-12 11:17,True,선정,0.112750,45.370056,
7,COPERNICUS/S2_SR_HARMONIZED/20260612T020651_20...,20260612T020651_20260612T021313_T52SEE,2026-06-12 11:17,False,제외: 기준 픽셀 결측,NaN,NaN,
8,COPERNICUS/S2_SR_HARMONIZED/20260722T020651_20...,20260722T020651_20260722T021351_T52SEE,2026-07-22 11:17,False,제외: 기준 픽셀 결측,NaN,NaN,
9,COPERNICUS/S2_SR_HARMONIZED/20260801T020651_20...,20260801T020651_20260801T021347_T52SDE,2026-08-01 11:17,True,선정,0.058519,34.974507,


지도 범례 설정

In [27]:
# NDCI 지도 색상
NDCI_STYLE = {
    "min": -0.3,
    "max": 0.5,
    "palette": [
        "2166ac",
        "67a9cf",
        "ffffbf",
        "a6d96a",
        "1a9850"
    ]
}

# 추정 클로로필-a 지도 색상
CHLA_STYLE = {
    "min": 0,
    "max": 100,
    "palette": [
        "313695",
        "74add1",
        "abd9e9",
        "ffffbf",
        "fdae61",
        "d73027"
    ]
}

# 편차 지도 색상
# 5 mg/m³ 단위로 구분
DEVIATION_COLORS = [
    "053061",
    "2166ac",
    "92c5de",
    "f7f7f7",
    "f4a582",
    "d6604d",
    "67001f"
]

# 편차 범례
DEVIATION_LEGEND = {
    "D < -15": "053061",
    "-15 ≤ D < -10": "2166ac",
    "-10 ≤ D < -5": "92c5de",
    "-5 ≤ D ≤ 5": "f7f7f7",
    "5 < D ≤ 10": "f4a582",
    "10 < D ≤ 15": "d6604d",
    "D > 15": "67001f"
}


def classify_deviation(deviation):
    # 편차를 5 mg/m³ 단위로 분류
    classes = deviation.multiply(0).add(3)

    return (
        classes
        .where(deviation.lt(-15), 0)
        .where(
            deviation.gte(-15).And(deviation.lt(-10)),
            1
        )
        .where(
            deviation.gte(-10).And(deviation.lt(-5)),
            2
        )
        .where(
            deviation.gt(5).And(deviation.lte(10)),
            4
        )
        .where(
            deviation.gt(10).And(deviation.lte(15)),
            5
        )
        .where(deviation.gt(15), 6)
        .toByte()
        .rename("Deviation_Class")
    )


print("지도 범례 준비 완료")
print("편차 구간: 5 mg/m³ 단위")

지도 범례 준비 완료
편차 구간: 5 mg/m³ 단위


**지도 표시**

영상을 선택하거나 지도 종류를 바꾸면 자동으로 갱신됩니다. 기준 픽셀은 노란 경계, 현재 채수지점은 마커로 표시됩니다.

In [28]:
# 선정된 영상만 선택 목록에 표시
image_options = [
    (
        f"{int(row.No):02d} | {row.Date_Time_KST} KST"
        f" | {row.MGRS_Tile}"
        f" | 구름 {row.Cloud_Percent_Global:.1f}%"
        f" | 기준 농도 {row.Site_Pixel_Chla_mg_m3:.2f} mg/m³",
        index
    )
    for index, row in image_table.iterrows()
]

image_selector = widgets.Dropdown(
    options=image_options,
    description="위성영상:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "initial"}
)

mode_selector = widgets.ToggleButtons(
    options=[
        ("NDCI", "ndci"),
        ("추정 클로로필-a", "chla"),
        ("채수지점 대비 편차", "deviation")
    ],
    value="deviation",
    description="지도:"
)

map_button = widgets.Button(
    description="지도 다시 표시",
    button_style="success"
)

main_output = widgets.Output()


def get_result_by_image_id(image_id):
    # 영상 ID를 기준으로 결과를 가져옴
    # image_table의 행 번호와 candidate_image_table의
    # 행 번호가 달라져도 올바른 영상 결과를 사용
    image_id = str(image_id)

    if not image_id.startswith(COLLECTION_ID):
        image_id = f"{COLLECTION_ID}/{image_id}"

    # 이미 계산한 영상은 저장된 결과 사용
    if image_id not in analysis_cache:
        result = process_scene(image_id)
        analysis_cache[image_id] = result

    return analysis_cache[image_id]


def draw_map(_=None):
    # 처리 중에는 선택을 잠금
    map_button.disabled = True
    image_selector.disabled = True
    mode_selector.disabled = True

    index = image_selector.value
    mode = mode_selector.value
    row = image_table.loc[index].copy()

    with main_output:
        clear_output(wait=True)
        print("선택 영상 분석 중:", row.Image_Index)

    try:
        # 영상 ID 기준으로 결과 가져오기
        result = get_result_by_image_id(
            row["Image_ID"]
        )

        # 기준 픽셀이 유효하지 않으면 지도 생성 중단
        if not result["site_valid"]:
            with main_output:
                clear_output(wait=True)

                print("기준 픽셀 결측: 지도 표시 불가")
                print(f"영상: {row.Image_Index}")
                print(
                    "이 영상은 4단계 선정 목록에 "
                    "포함되지 않아야 합니다."
                )

            return

        with main_output:
            clear_output(wait=True)

            print(f"촬영 시각: {row.Date_Time_KST} KST")
            print(f"영상: {row.Image_Index}")
            print(
                f"위성 / 타일: "
                f"{row.Satellite} / {row.MGRS_Tile}"
            )
            print(
                "기준: 현재 채수지점 좌표가 속한 20m 픽셀 1개"
            )

            print(
                f"기준 픽셀 NDCI: "
                f"{result['site_ndci']:.4f}"
            )
            print(
                f"기준 픽셀 추정 Chl-a: "
                f"{result['site_chla']:.2f} mg/m³"
            )

            if result["site_chla"] < 0:
                print(
                    "주의: 기준 픽셀의 추정 농도가 음수입니다. "
                    "값을 0으로 바꾸지 않고 그대로 보관합니다."
                )

            # 지도 생성
            m = geemap.Map(
                center=[SITE_LAT, SITE_LON],
                zoom=14,
                height="650px"
            )

            m.add_basemap("Esri.WorldImagery")

            # 선택 영상의 RGB
            rgb = (
                result["raw"]
                .select(["B4", "B3", "B2"])
                .multiply(0.0001)
                .clip(roi)
            )

            m.addLayer(
                rgb,
                {"min": 0, "max": 0.2},
                "선택 촬영일 Sentinel-2 RGB",
                False
            )

            # NDCI 지도
            if mode == "ndci":
                m.addLayer(
                    result["ndci"],
                    NDCI_STYLE,
                    "NDCI [무차원]",
                    True,
                    0.85
                )

                m.add_colorbar(
                    vis_params=NDCI_STYLE,
                    label="NDCI [dimensionless]",
                    position="bottomright"
                )

            # 추정 클로로필-a 지도
            elif mode == "chla":
                m.addLayer(
                    result["chla"],
                    CHLA_STYLE,
                    "추정 클로로필-a [mg/m³]",
                    True,
                    0.85
                )

                m.add_colorbar(
                    vis_params=CHLA_STYLE,
                    label="Estimated Chl-a [mg/m³]",
                    position="bottomright"
                )

            # 편차 지도
            else:
                m.addLayer(
                    classify_deviation(result["deviation"]),
                    {
                        "min": 0,
                        "max": 6,
                        "palette": DEVIATION_COLORS
                    },
                    "채수지점 픽셀 대비 편차 [mg/m³]",
                    True,
                    0.85
                )

                m.add_legend(
                    title="편차 D [mg/m³]",
                    legend_dict=DEVIATION_LEGEND,
                    position="bottomright",
                    layer_keys=[
                        "채수지점 픽셀 대비 편차 [mg/m³]"
                    ],
                    style={
                        "position": "absolute",
                        "z_index": 9999999,
                        "border": "1px solid gray",
                        "background": "rgba(255, 255, 255, 0.85)",
                        "padding": "5px",
                        "fontSize": "10px",
                        "lineHeight": "14px",
                        "width": "170px",
                        "maxHeight": "180px",
                        "overflow": "auto"
                    }
                )

                print(
                    "D = 주변 픽셀 추정 농도 "
                    "− 채수지점 픽셀 추정 농도"
                )
                print(
                    "빨강: 주변이 더 높음 / 파랑: 주변이 더 낮음"
                )
                print("편차 구간: 5 mg/m³ 단위")

            # 기준 픽셀 경계
            pixel_outline = ee.FeatureCollection([
                ee.Feature(result["site_pixel_geometry"])
            ]).style(
                color="FFFF00",
                fillColor="00000000",
                width=3
            )

            m.addLayer(
                pixel_outline,
                {},
                "기준 픽셀 경계: 단일 픽셀",
                True
            )

            # 현재 채수지점 마커
            marker = ipyleaflet.Marker(
                location=(SITE_LAT, SITE_LON),
                title="현재 New 채수지점",
                draggable=False
            )

            popup_text = (
                "<b>현재 New 채수지점</b><br>"
                f"경도: {SITE_LON:.8f}<br>"
                f"위도: {SITE_LAT:.8f}<br>"
                "추출 방식: 좌표가 속한 픽셀 1개<br>"
                f"NDCI: {result['site_ndci']:.4f}<br>"
                f"추정 Chl-a: "
                f"{result['site_chla']:.2f} mg/m³"
            )

            marker.popup = widgets.HTML(
                value=popup_text
            )

            m.add_layer(marker)

            # 연구지역 경계
            boundary = ee.FeatureCollection([
                ee.Feature(roi)
            ]).style(
                color="FF0000",
                fillColor="00000000",
                width=2
            )

            m.addLayer(
                boundary,
                {},
                "연구지역 ROI",
                True
            )

            # 실제로 표시한 지도를 저장 대상으로 보관
            saved_map_state.clear()

            saved_map_state.update({
                "map": m,
                "result": result,
                "row": row.copy(),
                "mode": mode
            })

            display(m)

    except Exception as error:
        with main_output:
            clear_output(wait=True)
            print("지도 표시 실패:", error)
            traceback.print_exc()

    finally:
        map_button.disabled = False
        image_selector.disabled = False
        mode_selector.disabled = False


# 선택 변경 시 자동 갱신
def on_selection_change(change):
    if change["name"] == "value":
        draw_map()


image_selector.observe(
    on_selection_change,
    names="value"
)

mode_selector.observe(
    on_selection_change,
    names="value"
)

# 수동 갱신 버튼
map_button.on_click(draw_map)

display(widgets.VBox([
    widgets.HTML(
        "<p>영상을 선택하거나 지도 종류를 변경하면 "
        "자동으로 갱신됩니다.</p>"
    ),
    image_selector,
    mode_selector,
    map_button,
    main_output
]))

draw_map()

지도 저장

In [34]:
# PNG 저장 버튼
png_save_button = widgets.Button(
    description="현재 분석 지도 PNG 저장",
    button_style="success"
)

save_all_png_button = widgets.Button(
    description="모든 영상 PNG 자동 저장",
    button_style="warning"
)

png_save_output = widgets.Output()

# 추정 Chl-a 표시 범위
CHLA_PNG_MIN = 0
CHLA_PNG_MAX = 60

# NDCI 표시 범위를 더 좁게 설정
NDCI_PNG_MIN = -0.05
NDCI_PNG_MAX = 0.25

# 단위 표기: 3을 위첨자로 크게 표시
UNIT_MG_M3 = r"mg/m$^{3}$"


def safe_name(text):
    return (
        str(text)
        .replace("/", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )


def get_displayed_map():
    if not saved_map_state:
        raise ValueError(
            "먼저 6단계에서 지도를 정상적으로 표시해 주세요."
        )

    return saved_map_state.copy()


def get_png_style(mode):
    if mode == "ndci":
        return (
            {
                "min": NDCI_PNG_MIN,
                "max": NDCI_PNG_MAX,
                "palette": NDCI_STYLE["palette"]
            },
            "NDCI",
            "NDCI [dimensionless]"
        )

    if mode == "chla":
        return (
            {
                "min": CHLA_PNG_MIN,
                "max": CHLA_PNG_MAX,
                "palette": CHLA_STYLE["palette"]
            },
            "추정 Chl-a",
            f"추정 Chl-a [{UNIT_MG_M3}]"
        )

    return (
        {
            "min": 0,
            "max": 6,
            "palette": DEVIATION_COLORS
        },
        "채수지점 픽셀 대비 편차",
        f"편차 [{UNIT_MG_M3}]"
    )


def get_analysis_image(result, mode):
    if mode == "ndci":
        return result["ndci"]

    if mode == "chla":
        return result["chla"]

    return classify_deviation(
        result["deviation"]
    )


def create_png(row, result, mode):
    style, title, legend = get_png_style(mode)

    image = get_analysis_image(result, mode)

    west = SITE_LON - lon_offset
    east = SITE_LON + lon_offset
    south = SITE_LAT - lat_offset
    north = SITE_LAT + lat_offset

    fig, ax = plt.subplots(figsize=(12, 10))

    # 배경: 해당 촬영일 Sentinel-2 RGB
    background_url = (
        result["raw"]
        .select(["B4", "B3", "B2"])
        .multiply(0.0001)
        .visualize(min=0, max=0.2)
        .getThumbURL({
            "region": roi,
            "dimensions": 1800,
            "crs": "EPSG:4326",
            "format": "png"
        })
    )

    background = Image.open(
        io.BytesIO(
            requests.get(
                background_url,
                timeout=180
            ).content
        )
    ).convert("RGBA")

    ax.imshow(
        background,
        extent=[west, east, south, north],
        origin="upper",
        zorder=1
    )

    # 분석 레이어
    analysis_url = image.visualize(**style).getThumbURL({
        "region": roi,
        "dimensions": 1800,
        "crs": "EPSG:4326",
        "format": "png"
    })

    analysis = Image.open(
        io.BytesIO(
            requests.get(
                analysis_url,
                timeout=180
            ).content
        )
    ).convert("RGBA")

    ax.imshow(
        analysis,
        extent=[west, east, south, north],
        origin="upper",
        alpha=0.75,
        zorder=2
    )

    # 채수지점
    ax.scatter(
        SITE_LON,
        SITE_LAT,
        s=110,
        marker="o",
        facecolor="yellow",
        edgecolor="black",
        linewidth=1.5,
        zorder=5,
        label="현재 채수지점"
    )

    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_aspect(
        1 / np.cos(np.deg2rad(SITE_LAT))
    )

    ax.set_xlabel("경도 [도]", fontsize=12)
    ax.set_ylabel("위도 [도]", fontsize=12)

    ax.set_title(
        f"{title}\n"
        f"{row['Date_Time_KST']} KST\n"
        f"기준 픽셀 추정 Chl-a: "
        f"{result['site_chla']:.2f} {UNIT_MG_M3}",
        fontsize=15,
        fontweight="bold"
    )

    # 범례
    if mode == "deviation":
        cmap = mcolors.ListedColormap([
            "#" + color
            for color in DEVIATION_COLORS
        ])

        norm = mcolors.BoundaryNorm(
            np.arange(-0.5, 7.5, 1),
            cmap.N
        )

    elif mode == "ndci":
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "ndci_colors",
            [
                "#" + color
                for color in NDCI_STYLE["palette"]
            ]
        )

        norm = mcolors.Normalize(
            vmin=NDCI_PNG_MIN,
            vmax=NDCI_PNG_MAX
        )

    else:
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "chla_colors",
            [
                "#" + color
                for color in CHLA_STYLE["palette"]
            ]
        )

        norm = mcolors.Normalize(
            vmin=CHLA_PNG_MIN,
            vmax=CHLA_PNG_MAX
        )

    scalar = plt.cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )
    scalar.set_array([])

    colorbar = fig.colorbar(
        scalar,
        ax=ax,
        fraction=0.05,
        pad=0.03
    )

    colorbar.set_label(
        legend,
        fontsize=16,
        fontweight="bold"
    )
    colorbar.ax.tick_params(labelsize=13)

    if mode == "deviation":
        # 일반 하이픈 사용: 글꼴 호환성 향상
        colorbar.set_ticks(range(7))
        colorbar.set_ticklabels([
            "D < -15",
            "-15 ≤ D < -10",
            "-10 ≤ D < -5",
            "-5 ≤ D ≤ 5",
            "5 < D ≤ 10",
            "10 < D ≤ 15",
            "D > 15"
        ])
        colorbar.ax.tick_params(labelsize=11)

    if mode == "ndci":
        colorbar.set_ticks(
            np.linspace(
                NDCI_PNG_MIN,
                NDCI_PNG_MAX,
                7
            )
        )

    if mode == "chla":
        colorbar.set_ticks(
            np.linspace(
                CHLA_PNG_MIN,
                CHLA_PNG_MAX,
                7
            )
        )

    ax.legend(
        loc="upper right",
        fontsize=12,
        framealpha=0.9
    )

    note = (
        "기준: 채수지점 좌표가 속한 단일 픽셀"
        " | 회색: 분석값이 없는 영역"
    )

    if mode == "deviation":
        note += (
            "\n편차 = 주변 픽셀 추정 Chl-a"
            " - 채수지점 픽셀 추정 Chl-a"
            f"\n편차 구간: 5 {UNIT_MG_M3} 단위"
        )

    if mode == "chla":
        note += (
            f"\n표시 범위: {CHLA_PNG_MIN}–"
            f"{CHLA_PNG_MAX} {UNIT_MG_M3}"
        )

    if mode == "ndci":
        note += (
            f"\n표시 범위: {NDCI_PNG_MIN}–"
            f"{NDCI_PNG_MAX}"
        )

    fig.text(
        0.5, 0.025,
        note,
        ha="center",
        fontsize=11
    )

    fig.tight_layout(rect=[0, 0.07, 1, 1])

    return fig


def save_current_png(_=None):
    png_save_button.disabled = True
    fig = None

    with png_save_output:
        clear_output(wait=True)

        try:
            state = get_displayed_map()
            row = state["row"]
            mode = state["mode"]
            result = state["result"]

            if not result["site_valid"]:
                raise ValueError(
                    "기준 픽셀이 유효하지 않습니다."
                )

            print("현재 분석 지도 PNG 생성 중")

            fig = create_png(row, result, mode)

            filename = OUTPUT_DIR / (
                f"분석지도_"
                f"{safe_name(row['Image_Index'])}"
                f"_{mode}.png"
            )

            fig.savefig(
                filename,
                dpi=300,
                bbox_inches="tight",
                facecolor="white"
            )

            plt.close(fig)
            fig = None

            print("현재 분석 지도 PNG 저장 완료")
            print(filename.name)
            print("배경: 해당 촬영일 Sentinel-2 RGB 포함")

            files.download(str(filename))

        except Exception as error:
            print("PNG 저장 실패:", error)
            traceback.print_exc()

        finally:
            if fig is not None:
                plt.close(fig)

            png_save_button.disabled = False


def save_all_png(_=None):
    save_all_png_button.disabled = True
    fig = None

    with png_save_output:
        clear_output(wait=True)

        print("모든 선정 영상의 PNG 저장을 시작합니다.")
        print("각 영상마다 3종 지도를 저장합니다.\n")

        saved_files = []

        try:
            total = len(image_table)

            for number, (index, row) in enumerate(
                image_table.iterrows(), start=1
            ):
                print(
                    f"[{number}/{total}] "
                    f"{row['Date_Time_KST']} | "
                    f"{row['Image_Index']}"
                )

                result = get_result_by_image_id(
                    row["Image_ID"]
                )

                if not result["site_valid"]:
                    print("  기준 픽셀 결측: 저장 생략")
                    continue

                for mode in [
                    "ndci",
                    "chla",
                    "deviation"
                ]:
                    fig = create_png(
                        row,
                        result,
                        mode
                    )

                    filename = OUTPUT_DIR / (
                        f"분석지도_"
                        f"{safe_name(row['Image_Index'])}"
                        f"_{mode}.png"
                    )

                    fig.savefig(
                        filename,
                        dpi=300,
                        bbox_inches="tight",
                        facecolor="white"
                    )

                    plt.close(fig)
                    fig = None

                    saved_files.append(filename)

                    print(f"  저장 완료: {mode}")

            # ZIP으로 묶기
            import zipfile

            zip_name = OUTPUT_DIR / "모든_영상_PNG.zip"

            with zipfile.ZipFile(
                zip_name,
                "w",
                zipfile.ZIP_DEFLATED
            ) as zip_file:
                for file_path in saved_files:
                    zip_file.write(
                        file_path,
                        arcname=file_path.name
                    )

            print("\n모든 영상 PNG 저장 완료")
            print(f"저장 파일 수: {len(saved_files)}")
            print(f"ZIP 파일: {zip_name.name}")

            files.download(str(zip_name))

        except Exception as error:
            print("전체 PNG 저장 실패:", error)
            traceback.print_exc()

        finally:
            if fig is not None:
                plt.close(fig)

            save_all_png_button.disabled = False


png_save_button.on_click(save_current_png)
save_all_png_button.on_click(save_all_png)

display(
    widgets.VBox([
        widgets.HTML(
            "<p><b>PNG 저장 방식</b><br>"
            "1. 현재 분석 지도 PNG 저장<br>"
            "2. 모든 영상 PNG 자동 저장</p>"
        ),
        widgets.HBox([
            png_save_button,
            save_all_png_button
        ]),
        png_save_output
    ])
)

5월 13일 추정 chl-a농도만 -167.93으로 이상치가 나와 원인 분석

In [38]:
# 문제 날짜 확인
problem_image_index = (
    "20260513T020651_20260513T021205_T52SDE"
    .strip()
)

print("찾으려는 영상 이름:")
print(problem_image_index)

# Image_Index 열에서 일치하는 행 찾기
problem_row = candidate_image_table.loc[
    candidate_image_table["Image_Index"]
    == problem_image_index
]

if problem_row.empty:
    print("\n해당 영상 이름이 목록에 없습니다.")
    print("비슷한 이름이 있는지 확인합니다.\n")

    similar = candidate_image_table[
        candidate_image_table["Image_Index"]
        .astype(str)
        .str.contains("20260513", na=False)
    ]

    display(similar[[
        "Image_ID",
        "Image_Index",
        "Date_Time_KST"
    ]])

    raise ValueError(
        "입력한 Image_Index가 목록에 없습니다. "
        "위 목록에서 정확한 이름을 확인하세요."
    )

problem_row = problem_row.iloc[0]

print("\n찾은 영상 정보")
display(problem_row[[
    "Image_ID",
    "Image_Index",
    "Date_Time_KST",
    "MGRS_Tile",
    "Cloud_Percent_Global"
]])

problem_result = get_result_by_image_id(
    problem_row["Image_ID"]
)

problem_image = ee.Image(problem_row["Image_ID"])
grid = problem_result["grid"]

print("\n기준 픽셀 분석 결과")
print("기준 픽셀 NDCI:", problem_result["site_ndci"])
print(
    "기준 픽셀 추정 Chl-a:",
    problem_result["site_chla"],
    "mg/m³"
)

# 채수지점 픽셀의 원래 B4·B5·SCL 값
raw_values = problem_image.select([
    "B4", "B5", "SCL"
]).reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=site_point,
    scale=20,
    maxPixels=1000
).getInfo()

print("\n채수지점 픽셀의 원래 값")
print(raw_values)

b4 = raw_values.get("B4")
b5 = raw_values.get("B5")
scl = raw_values.get("SCL")

if b4 is not None and b5 is not None:
    b4_reflectance = float(b4) * 0.0001
    b5_reflectance = float(b5) * 0.0001

    denominator = b4_reflectance + b5_reflectance

    print("\n반사도로 변환한 값")
    print(f"B4 반사도: {b4_reflectance:.6f}")
    print(f"B5 반사도: {b5_reflectance:.6f}")
    print(f"B4 + B5: {denominator:.6f}")

    if denominator != 0:
        ndci_value = (
            (b5_reflectance - b4_reflectance)
            / denominator
        )

        print(f"NDCI: {ndci_value:.4f}")

        chla_value = (
            ndci_value * SLOPE + INTERCEPT
        )

        print(
            f"추정 Chl-a: {chla_value:.2f} mg/m³"
        )

if scl is not None:
    print(f"\nSCL: {scl}")

    if scl == 6:
        print("SCL = 6: 수면으로 분류됨")
    elif scl in [3, 8, 9, 10]:
        print("SCL = 구름 그림자 또는 구름 관련 클래스")
    else:
        print("SCL = 수면이 아닌 클래스")

찾으려는 영상 이름:
20260513T020651_20260513T021205_T52SDE

찾은 영상 정보


,2
Image_ID,COPERNICUS/S2_SR_HARMONIZED/20260513T020651_20...
Image_Index,20260513T020651_20260513T021205_T52SDE
Date_Time_KST,2026-05-13 11:17
MGRS_Tile,52SDE
Cloud_Percent_Global,1.342889



기준 픽셀 분석 결과
기준 픽셀 NDCI: -1.0
기준 픽셀 추정 Chl-a: -167.931789 mg/m³

채수지점 픽셀의 원래 값
{'B4': 25, 'B5': 0, 'SCL': 6}

반사도로 변환한 값
B4 반사도: 0.002500
B5 반사도: 0.000000
B4 + B5: 0.002500
NDCI: -1.0000
추정 Chl-a: -167.93 mg/m³

SCL: 6
SCL = 6: 수면으로 분류됨
